# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

**Choix pour ce lab : PySpark comme langage principal pour tout le Part 1 et le Part 2.**
La Partie 3 réutilise 3 de ces questions en pur Spark SQL, dont au moins une avec un `JOIN`.

### 1.1 — Clé unique par trajet

In [5]:
from pyspark.sql import functions as F

# monotonically_increasing_id() garantit un identifiant unique et croissant par ligne,
# même sur un DataFrame distribué (contrairement à un simple index pandas)
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())

df_trips.select("trip_id").show(5)

+-----------+
|    trip_id|
+-----------+
|60129542144|
|60129542145|
|60129542146|
|60129542147|
|60129542148|
+-----------+
only showing top 5 rows


### 1.2 — Trajet avec le plus de passagers

In [6]:
df_trips.orderBy(F.col("passenger_count").desc()) \
    .select("trip_id", "passenger_count", "tpep_pickup_datetime", "PULocationID", "DOLocationID") \
    .show(5)

+-----------+---------------+--------------------+------------+------------+
|    trip_id|passenger_count|tpep_pickup_datetime|PULocationID|DOLocationID|
+-----------+---------------+--------------------+------------+------------+
|60131554242|            9.0| 2019-01-10 00:43:10|          68|          68|
|60136828827|            9.0| 2019-01-30 18:34:12|         236|         236|
|60132426139|            9.0| 2019-01-13 04:13:24|         263|         263|
|60130838431|            9.0| 2019-01-07 03:19:36|         163|         163|
|60134076851|            9.0| 2019-01-19 16:45:25|           1|           1|
+-----------+---------------+--------------------+------------+------------+
only showing top 5 rows


### 1.3 — Nombre moyen de passagers

In [7]:
df_trips.select(F.avg("passenger_count").alias("avg_passenger_count")).show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



### 1.4 — Trajet le plus court / le plus long (distance et durée)

In [8]:
# On exclut les distances nulles/négatives qui ne représentent pas de vrais trajets
print("Trajet le plus court (distance) :")
df_trips.filter(F.col("trip_distance") > 0) \
    .orderBy(F.col("trip_distance").asc()) \
    .select("trip_id", "trip_distance").show(5)

print("Trajet le plus long (distance) :")
df_trips.orderBy(F.col("trip_distance").desc()) \
    .select("trip_id", "trip_distance").show(5)

Trajet le plus court (distance) :
+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|60129560972|         0.01|
|60129564611|         0.01|
|60129560973|         0.01|
|60129554708|         0.01|
|60129562361|         0.01|
+-----------+-------------+
only showing top 5 rows
Trajet le plus long (distance) :
+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|60135616235|        831.8|
|60133828777|        700.7|
|60136313129|       214.01|
|60134249678|       211.36|
|60134423929|       201.27|
+-----------+-------------+
only showing top 5 rows


In [9]:
# Durée du trajet en secondes (dropoff - pickup)
df_trips = df_trips.withColumn(
    "trip_duration_sec",
    F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
)

print("Trajet le plus court (durée) :")
df_trips.filter(F.col("trip_duration_sec") > 0) \
    .orderBy(F.col("trip_duration_sec").asc()) \
    .select("trip_id", "trip_duration_sec").show(5)

print("Trajet le plus long (durée) :")
df_trips.orderBy(F.col("trip_duration_sec").desc()) \
    .select("trip_id", "trip_duration_sec").show(5)

Trajet le plus court (durée) :
+-----------+-----------------+
|    trip_id|trip_duration_sec|
+-----------+-----------------+
|60129573664|                1|
|60129609220|                1|
|60129575663|                1|
|60129565495|                1|
|60129585711|                1|
+-----------+-----------------+
only showing top 5 rows
Trajet le plus long (durée) :
+-----------+-----------------+
|    trip_id|trip_duration_sec|
+-----------+-----------------+
|60129610411|          2618881|
|60130134406|          2031401|
|60130418000|          1891926|
|60133257869|           431454|
|60131257409|           101222|
+-----------+-----------------+
only showing top 5 rows


### 1.5 — Jour le plus/moins chargé

In [10]:
df_trips = df_trips.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

daily_counts = df_trips.groupBy("pickup_date").count()

print("Jour le plus chargé :")
daily_counts.orderBy(F.col("count").desc()).show(5)

print("Jour le moins chargé :")
daily_counts.orderBy(F.col("count").asc()).show(5)

Jour le plus chargé :
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
| 2019-01-11|291714|
| 2019-01-31|284625|
| 2019-01-17|284580|
| 2019-01-24|281959|
+-----------+------+
only showing top 5 rows
Jour le moins chargé :
+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2019-02-23|    1|
| 2019-05-20|    1|
| 2019-08-13|    1|
| 2019-07-23|    1|
| 2018-12-21|    1|
+-----------+-----+
only showing top 5 rows


### 1.6 — Heure / créneau de la journée le plus et le moins chargé

In [11]:
df_trips = df_trips.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

# Bucket morning / afternoon / evening / late_night
df_trips = df_trips.withColumn(
    "time_of_day",
    F.when((F.col("pickup_hour") >= 5)  & (F.col("pickup_hour") < 12), "morning")
     .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") < 17), "afternoon")
     .when((F.col("pickup_hour") >= 17) & (F.col("pickup_hour") < 21), "evening")
     .otherwise("late_night")
)

print("Par créneau :")
df_trips.groupBy("time_of_day").count().orderBy(F.col("count").desc()).show()

print("Par heure (top 5 / bottom 5) :")
hourly = df_trips.groupBy("pickup_hour").count()
hourly.orderBy(F.col("count").desc()).show(5)
hourly.orderBy(F.col("count").asc()).show(5)

Par créneau :
+-----------+-------+
|time_of_day|  count|
+-----------+-------+
|  afternoon|2111999|
|    morning|2035497|
|    evening|1882211|
| late_night|1666910|
+-----------+-------+

Par heure (top 5 / bottom 5) :
+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515390|
|         19|475186|
|         17|468479|
|         15|452691|
|         14|433139|
+-----------+------+
only showing top 5 rows
+-----------+------+
|pickup_hour| count|
+-----------+------+
|          4| 61424|
|          5| 75533|
|          3| 78086|
|          2|109421|
|          1|149254|
+-----------+------+
only showing top 5 rows


### 1.7 — Jour de la semaine le plus/moins chargé en moyenne

In [12]:
df_trips = df_trips.withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "EEEE"))

# nb de trajets par (jour de semaine, date) puis moyenne par jour de semaine
# -> évite de biaiser si un jour de la semaine (ex: lundi) apparaît plus souvent que les autres dans le mois
dow_avg = df_trips.groupBy("pickup_dow", "pickup_date").count() \
    .groupBy("pickup_dow") \
    .agg(F.avg("count").alias("avg_trips_per_day")) \
    .orderBy(F.col("avg_trips_per_day").desc())

dow_avg.show()

+----------+------------------+
|pickup_dow| avg_trips_per_day|
+----------+------------------+
|  Thursday| 193863.2857142857|
|    Friday|155316.42857142858|
|  Saturday|144283.57142857142|
| Wednesday|140584.88888888888|
|   Tuesday|          120908.4|
|    Sunday|        107488.125|
|    Monday|           90812.1|
+----------+------------------+



### 1.8 — Impact de la distance / du nombre de passagers sur le pourboire

In [13]:
corr_distance_tip = df_trips.stat.corr("trip_distance", "tip_amount")
corr_passenger_tip = df_trips.stat.corr("passenger_count", "tip_amount")

print(f"Corrélation distance <-> pourboire   : {corr_distance_tip:.4f}")
print(f"Corrélation passagers <-> pourboire  : {corr_passenger_tip:.4f}")

# Vue complémentaire : pourboire moyen par tranche de distance
df_trips.withColumn(
    "distance_bucket",
    F.when(F.col("trip_distance") < 1, "<1 mi")
     .when(F.col("trip_distance") < 3, "1-3 mi")
     .when(F.col("trip_distance") < 10, "3-10 mi")
     .otherwise("10+ mi")
).groupBy("distance_bucket").agg(F.avg("tip_amount").alias("avg_tip")).orderBy("distance_bucket").show()

Corrélation distance <-> pourboire   : 0.5269
Corrélation passagers <-> pourboire  : 0.0044
+---------------+------------------+
|distance_bucket|           avg_tip|
+---------------+------------------+
|         1-3 mi|1.4519057215266515|
|         10+ mi| 6.219343545352378|
|        3-10 mi| 2.779081022870883|
|          <1 mi|0.9403610547768468|
+---------------+------------------+



**Lecture des résultats :** une corrélation proche de 0 signifie qu'il n'y a pas de relation linéaire forte entre la distance/le nombre de passagers et le pourboire pris isolément — ce qui est cohérent avec le fait que beaucoup de clients arrondissent ou paient un pourcentage fixe indépendamment de la distance. Le tableau par tranche de distance permet de vérifier s'il existe malgré tout une tendance (ex : pourboire moyen plus élevé sur les longues courses).

### 1.9 — Frais "extra" le plus élevé

In [14]:
df_trips.orderBy(F.col("extra").desc()) \
    .select("trip_id", "extra", "tpep_pickup_datetime", "PULocationID", "DOLocationID") \
    .show(5)

+-----------+------+--------------------+------------+------------+
|    trip_id| extra|tpep_pickup_datetime|PULocationID|DOLocationID|
+-----------+------+--------------------+------------+------------+
|60134865627|535.38| 2019-01-23 08:58:09|          24|         264|
|60136995374| 23.04| 2019-01-31 10:06:09|         237|         264|
|60129853196|  18.5| 2019-01-02 16:33:28|         233|           1|
|60131997230|  18.5| 2019-01-11 16:08:48|         161|           1|
|60129676693|  18.5| 2019-01-01 16:09:32|         261|           1|
+-----------+------+--------------------+------------+------------+
only showing top 5 rows


### 1.10 — Détection d'outliers

In [15]:
outliers = df_trips.filter(
    (F.col("trip_duration_sec") <= 0) |          # dropoff avant ou égal au pickup -> impossible
    (F.col("trip_distance") > 100) |             # distance aberrante pour un trajet urbain
    (F.col("fare_amount") > 500) |               # tarif largement hors norme
    (F.col("fare_amount") < 0) |                 # tarif négatif -> erreur de saisie/remboursement mal codé
    (F.col("passenger_count") > 6) |             # capacité max légale d'un taxi jaune
    (F.col("passenger_count") == 0)              # aucun passager déclaré
)

outliers.select(
    "trip_id", "trip_distance", "trip_duration_sec",
    "fare_amount", "passenger_count", "extra"
).show(20, truncate=False)

print(f"Nombre de trajets suspects : {outliers.count()} sur {df_trips.count()}")

+-----------+-------------+-----------------+-----------+---------------+-----+
|trip_id    |trip_distance|trip_duration_sec|fare_amount|passenger_count|extra|
+-----------+-------------+-----------------+-----------+---------------+-----+
|60129542172|0.0          |0                |6.5        |3.0            |0.5  |
|60129542300|5.3          |57               |2.5        |0.0            |0.5  |
|60129542372|18.0         |1858             |52.0       |0.0            |0.0  |
|60129542373|8.9          |1879             |29.5       |0.0            |0.5  |
|60129542442|1.0          |634              |8.0        |0.0            |0.5  |
|60129542807|0.1          |39               |-2.5       |2.0            |-0.5 |
|60129542825|0.0          |0                |2.5        |1.0            |0.5  |
|60129543049|0.7          |387              |6.0        |0.0            |0.5  |
|60129543304|0.9          |251              |5.0        |0.0            |0.5  |
|60129543305|2.2          |645          

**Raisonnement sur les outliers :** plusieurs types d'anomalies ressortent des données de ce type de jeu :

- **Durées négatives ou nulles** : le dropoff est antérieur (ou égal) au pickup, ce qui est physiquement impossible — probablement une erreur d'horodatage du compteur.
- **Distances extrêmes (> 100 miles)** pour un trajet censé rester dans/autour de NYC : soit une erreur GPS/compteur, soit un trajet longue distance mal catégorisé.
- **Tarifs (`fare_amount`) très élevés (> 500 $) ou négatifs** : les valeurs négatives correspondent souvent à des annulations/remboursements enregistrés comme des trajets classiques ; les valeurs très élevées peuvent être des doubles facturations ou des erreurs de saisie.
- **`passenger_count` = 0 ou > 6** : un taxi jaune standard ne peut légalement pas transporter plus de 4 à 6 passagers selon le véhicule ; un compteur à 0 passager indique probablement un capteur non déclaré ou un trajet test.

Dans un pipeline de production, ces lignes seraient soit filtrées, soit traitées séparément (ex: valeurs nulles horodatage) plutôt que supprimées à l'aveugle, pour ne pas biaiser les statistiques globales calculées plus haut.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

### 2.1 — Chargement de la table des zones (boroughs)

In [16]:
zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(zone_lookup_url)
zone_lookup_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_lookup_file, "wb") as f:
        f.write(response.content)

df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_lookup_file)

df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


### 2.2 — Jointure pickup / dropoff avec les boroughs

In [17]:
# Jointure sur PULocationID pour les pickups, sur DOLocationID pour les dropoffs
df_pu = df_trips.join(
    df_zones.select("LocationID", "Borough"),
    df_trips.PULocationID == df_zones.LocationID,
    "left"
).withColumnRenamed("Borough", "PU_Borough")

df_do = df_trips.join(
    df_zones.select("LocationID", "Borough"),
    df_trips.DOLocationID == df_zones.LocationID,
    "left"
).withColumnRenamed("Borough", "DO_Borough")

### 2.3 — Borough avec le plus de pickups / dropoffs

In [18]:
print("Pickups par borough :")
df_pu.groupBy("PU_Borough").count().orderBy(F.col("count").desc()).show()

print("Dropoffs par borough :")
df_do.groupBy("DO_Borough").count().orderBy(F.col("count").desc()).show()

Pickups par borough :
+-------------+-------+
|   PU_Borough|  count|
+-------------+-------+
|    Manhattan|6950965|
|       Queens| 471173|
|      Unknown| 159815|
|     Brooklyn|  91905|
|        Bronx|  18062|
|          N/A|   3890|
|          EWR|    446|
|Staten Island|    361|
+-------------+-------+

Dropoffs par borough :
+-------------+-------+
|   DO_Borough|  count|
+-------------+-------+
|    Manhattan|6817355|
|       Queens| 340972|
|     Brooklyn| 301105|
|      Unknown| 149097|
|        Bronx|  58085|
|          N/A|  16904|
|          EWR|  10914|
|Staten Island|   2185|
+-------------+-------+



### 2.4 — Heures chargées/creuses par borough

In [19]:
df_pu.groupBy("PU_Borough", "pickup_hour").count() \
    .orderBy("PU_Borough", F.col("count").desc()) \
    .show(30)

+----------+-----------+-----+
|PU_Borough|pickup_hour|count|
+----------+-----------+-----+
|     Bronx|          7| 1803|
|     Bronx|          8| 1445|
|     Bronx|          6| 1301|
|     Bronx|          9| 1158|
|     Bronx|         10| 1079|
|     Bronx|         14| 1014|
|     Bronx|         12|  956|
|     Bronx|         13|  918|
|     Bronx|         15|  897|
|     Bronx|         11|  885|
|     Bronx|         17|  812|
|     Bronx|         16|  756|
|     Bronx|         18|  741|
|     Bronx|          5|  736|
|     Bronx|         19|  519|
|     Bronx|         20|  440|
|     Bronx|         23|  410|
|     Bronx|         21|  408|
|     Bronx|          4|  403|
|     Bronx|         22|  353|
|     Bronx|          0|  324|
|     Bronx|          1|  254|
|     Bronx|          3|  225|
|     Bronx|          2|  225|
|  Brooklyn|          8| 6935|
|  Brooklyn|          7| 6318|
|  Brooklyn|          9| 4691|
|  Brooklyn|          6| 4456|
|  Brooklyn|         23| 4342|
|  Brook

### 2.5 — Jours de la semaine les plus chargés par borough

In [20]:
df_pu.groupBy("PU_Borough", "pickup_dow").count() \
    .orderBy("PU_Borough", F.col("count").desc()) \
    .show(30)

+----------+----------+-------+
|PU_Borough|pickup_dow|  count|
+----------+----------+-------+
|     Bronx|  Thursday|   3121|
|     Bronx|   Tuesday|   3059|
|     Bronx| Wednesday|   2999|
|     Bronx|    Friday|   2666|
|     Bronx|    Monday|   2177|
|     Bronx|    Sunday|   2112|
|     Bronx|  Saturday|   1928|
|  Brooklyn|   Tuesday|  15779|
|  Brooklyn|  Thursday|  15714|
|  Brooklyn| Wednesday|  15101|
|  Brooklyn|    Friday|  13092|
|  Brooklyn|  Saturday|  11604|
|  Brooklyn|    Sunday|  11099|
|  Brooklyn|    Monday|   9516|
|       EWR| Wednesday|     83|
|       EWR|   Tuesday|     77|
|       EWR|    Friday|     74|
|       EWR|    Sunday|     68|
|       EWR|  Thursday|     58|
|       EWR|  Saturday|     55|
|       EWR|    Monday|     31|
| Manhattan|  Thursday|1229554|
| Manhattan| Wednesday|1144782|
| Manhattan|   Tuesday|1086202|
| Manhattan|    Friday| 984950|
| Manhattan|  Saturday| 927504|
| Manhattan|    Monday| 807748|
| Manhattan|    Sunday| 770225|
|       

### 2.6 — Distance moyenne par borough

In [21]:
df_pu.groupBy("PU_Borough") \
    .agg(F.avg("trip_distance").alias("avg_trip_distance")) \
    .orderBy(F.col("avg_trip_distance").desc()) \
    .show()

+-------------+------------------+
|   PU_Borough| avg_trip_distance|
+-------------+------------------+
|Staten Island|12.503601108033246|
|       Queens|11.283218499361993|
|        Bronx| 7.233194552098303|
|     Brooklyn| 4.787677275447492|
|          N/A| 3.193850899742941|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|    Manhattan|2.2286693358402596|
+-------------+------------------+



### 2.7 — Tarif moyen par borough

In [22]:
df_pu.groupBy("PU_Borough") \
    .agg(F.avg("fare_amount").alias("avg_fare_amount")) \
    .orderBy(F.col("avg_fare_amount").desc()) \
    .show()

+-------------+------------------+
|   PU_Borough|   avg_fare_amount|
+-------------+------------------+
|          EWR| 76.24024663677126|
|          N/A|  59.5731593830335|
|Staten Island|45.289861495844896|
|       Queens| 35.14462651722029|
|        Bronx| 26.26890543682963|
|     Brooklyn|18.649132800172286|
|      Unknown|14.944423051653523|
|    Manhattan|10.792468572351568|
+-------------+------------------+



### 2.8 — Tarif le plus élevé / le plus bas, et borough associé

In [23]:
print("Tarif le plus élevé :")
df_pu.orderBy(F.col("fare_amount").desc()) \
    .select("trip_id", "fare_amount", "PU_Borough").show(5)

print("Tarif le plus bas :")
df_pu.orderBy(F.col("fare_amount").asc()) \
    .select("trip_id", "fare_amount", "PU_Borough").show(5)

Tarif le plus élevé :
+-----------+-----------+----------+
|    trip_id|fare_amount|PU_Borough|
+-----------+-----------+----------+
|60132041799|  623259.86| Manhattan|
|60134865627|  355676.98| Manhattan|
|60131702115|    36090.3|   Unknown|
|60131434925|   34674.65|   Unknown|
|60131191595|   33023.53|   Unknown|
+-----------+-----------+----------+
only showing top 5 rows
Tarif le plus bas :
+-----------+-----------+----------+
|    trip_id|fare_amount|PU_Borough|
+-----------+-----------+----------+
|60134432793|     -362.0|    Queens|
|60135850344|     -320.0|       N/A|
|60129599238|     -300.0|     Bronx|
|60136769998|     -300.0|       N/A|
|60130747032|     -284.0|    Queens|
+-----------+-----------+----------+
only showing top 5 rows


### 2.9 — Comparaison avec janvier 2025

In [24]:
download_url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

response = requests.get(download_url_2025)
jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

df_trips_2025 = spark.read.parquet(jan_2025_trip_data)

In [25]:
def metrics(df, label):
    return df.select(
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("tip_amount").alias("avg_tip"),
        F.avg("passenger_count").alias("avg_passengers")
    ).withColumn("period", F.lit(label))

metrics_2019 = metrics(df_trips, "2019-01")
metrics_2025 = metrics(df_trips_2025, "2025-01")

metrics_2019.unionByName(metrics_2025).show()

+------------------+-----------------+------------------+------------------+-------+
|      avg_distance|         avg_fare|           avg_tip|    avg_passengers| period|
+------------------+-----------------+------------------+------------------+-------+
|2.8301461681153532|12.52967677747685|1.8208300763883147|1.5670317144945614|2019-01|
| 5.855126178843539|17.08180276045484|2.9598127862758044|1.2978589658806226|2025-01|
+------------------+-----------------+------------------+------------------+-------+



**À commenter une fois les vrais chiffres obtenus :** noter ici si la distance moyenne, le tarif moyen, le pourboire moyen ou le nombre moyen de passagers ont significativement évolué entre janvier 2019 et janvier 2025 (inflation tarifaire, changement d'usage post-Covid, hausse du paiement par carte facilitant le pourboire, etc.).

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

Le travail principal a été fait en PySpark. On reprend ici **3 questions en Spark SQL pur** :

1. Nombre moyen de passagers
2. Frais "extra" le plus élevé
3. Pickups par borough (**avec `JOIN`**)

In [26]:
# Exposer les DataFrames comme vues SQL temporaires
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

### 3.1 — Nombre moyen de passagers (SQL)

In [27]:
spark.sql("""
    SELECT AVG(passenger_count) AS avg_passenger_count
    FROM trips
""").show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



### 3.2 — Frais "extra" le plus élevé (SQL)

In [28]:
spark.sql("""
    SELECT trip_id, extra, tpep_pickup_datetime
    FROM trips
    ORDER BY extra DESC
    LIMIT 5
""").show()

+-----------+------+--------------------+
|    trip_id| extra|tpep_pickup_datetime|
+-----------+------+--------------------+
|60134865627|535.38| 2019-01-23 08:58:09|
|60136995374| 23.04| 2019-01-31 10:06:09|
|60131997230|  18.5| 2019-01-11 16:08:48|
|60129853196|  18.5| 2019-01-02 16:33:28|
|60133559279|  18.5| 2019-01-17 16:24:12|
+-----------+------+--------------------+



### 3.3 — Pickups par borough (SQL + JOIN)

In [29]:
spark.sql("""
    SELECT z.Borough AS PU_Borough, COUNT(*) AS trip_count
    FROM trips t
    LEFT JOIN zones z
        ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY trip_count DESC
""").show()

+-------------+----------+
|   PU_Borough|trip_count|
+-------------+----------+
|    Manhattan|   6950965|
|       Queens|    471173|
|      Unknown|    159815|
|     Brooklyn|     91905|
|        Bronx|     18062|
|          N/A|      3890|
|          EWR|       446|
|Staten Island|       361|
+-------------+----------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

### Piste 1 — Compléter 2019 et calculer la saison la plus chargée

Squelette pour étendre le pipeline à toute l'année 2019 (à dérouler mois par mois puis `unionByName`) :

In [33]:
months_2019 = [f"{m:02d}" for m in range(1, 13)]
dfs = []

for m in months_2019:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-{m}.parquet"
    fname = f"yellow_tripdata_2019-{m}.parquet"
    resp = requests.get(url)
    if resp.status_code == 200:
        with open(fname, "wb") as f:
            f.write(resp.content)
        dfs.append(spark.read.parquet(fname))

df_2019_full = dfs[0]
for d in dfs[1:]:
    df_2019_full = df_2019_full.unionByName(d, allowMissingColumns=True)

df_2019_full = df_2019_full.withColumn(
    "season",
    F.when(F.month("tpep_pickup_datetime").isin(12, 1, 2), "winter")
     .when(F.month("tpep_pickup_datetime").isin(3, 4, 5), "spring")
     .when(F.month("tpep_pickup_datetime").isin(6, 7, 8), "summer")
     .otherwise("fall")
)

df_2019_full.groupBy("season").count().orderBy(F.col("count").desc()).show()

+------+--------+
|season|   count|
+------+--------+
|spring|22941027|
|winter|21643025|
|  fall|20659565|
|summer|19354827|
+------+--------+



### Piste 2 — Visualisations natives Spark (v4+) sur 3 questions

In [34]:
# Ces appels utilisent l'intégration de visualisation native des DataFrames Spark 4
# (nécessite un environnement compatible, ex: Databricks ou Spark Connect avec le rendu activé)

# 1. Nombre de trajets par heure de la journée
hourly.orderBy("pickup_hour").plot.bar(x="pickup_hour", y="count")

# 2. Nombre moyen de trajets par jour de la semaine
dow_avg.plot.bar(x="pickup_dow", y="avg_trips_per_day")

# 3. Tarif moyen par borough
df_pu.groupBy("PU_Borough").agg(F.avg("fare_amount").alias("avg_fare_amount")) \
    .plot.bar(x="PU_Borough", y="avg_fare_amount")

PySparkImportError: [PACKAGE_NOT_INSTALLED] Plotly >= 4.8 must be installed; however, it was not found.

### Piste 3 — Explorer un autre jeu de données

Idée de prolongement : croiser ce dataset avec un autre jeu de données ouvert (ex: météo NYC via NOAA, ou données de trafic) pour tester si les conditions météo influencent le volume de trajets ou les distances/tarifs moyens.